# Cell 1 — Install

In [ ]:
%%capture
!pip install openai datasets
print("Done!")

# Cell 2 — Config

In [ ]:
import os, json
import pandas as pd
from openai import OpenAI
from google.colab import drive

drive.mount('/content/drive')

from google.colab import userdata
GROQ_API_KEY = userdata.get("groq_api_key")

TEST_CSV    = "/content/drive/MyDrive/VisualStep/test_set_200.csv"
RESULTS_DIR = "/content/drive/MyDrive/VisualStep/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

SYSTEM_PROMPT = """You are an expert in ADHD classroom accommodations for primary school children aged 5-10.
Transform the teacher's spoken classroom instruction into an ADHD-adapted visual step card.

Rules:
- Maximum 5 steps
- Each step: one concrete physical action only
- Each action: maximum 6 words
- Start each step with an imperative verb
- Include a pictogram keyword in the icon field
- Output JSON only, no other text

Output format:
{"steps":[{"id":1,"action":"Open your book","detail":"page 23","icon":"book","check":true}],"n_steps":1,"support_level":"medium"}

support_level: "light" (1-2 steps), "medium" (3-4 steps), "high" (5 steps)"""

print("Config ready")

# Cell 3 — Load test set

In [ ]:
df_test = pd.read_csv(TEST_CSV)
print(f"Test set: {len(df_test)} examples")

# Cell 4 — Inference functions

In [ ]:
import time

def generate_step_card(instruction, year, subject, model_id, max_tokens=300, extra_body=None):
    user_content = f"Year: {year} | Subject: {subject} | Instruction: {instruction}"
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_content},
            ],
            temperature=0.1,
            max_tokens=max_tokens,
            extra_body=extra_body or {}
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  ERROR: {str(e)}")
        return ""

def run_baseline(model_id, model_name, output_filename, sleep=0.3, extra_body=None):
    print(f"\n{'='*55}")
    print(f"Running: {model_name}")
    print(f"{'='*55}")

    results = []
    ok = fail = 0

    for i, (_, row) in enumerate(df_test.iterrows()):
        if i % 10 == 0:
            print(f"  [{i+1}/{len(df_test)}]", end="", flush=True)

        output = generate_step_card(
            row['input'], row['year'], row['subject'],
            model_id, extra_body=extra_body
        )

        try:
            parsed     = json.loads(output)
            valid_json = True
            n_steps    = parsed.get('n_steps', len(parsed.get('steps', [])))
            support    = parsed.get('support_level', '')
            ok += 1
        except json.JSONDecodeError:
            valid_json = False
            n_steps    = 0
            support    = ''
            fail += 1

        results.append({
            'id':            row.get('id', f'TEST-{i}'),
            'source':        row.get('source', ''),
            'year':          row['year'],
            'subject':       row['subject'],
            'type':          row.get('type', ''),
            'input':         row['input'],
            'gold_output':   str(row.get('output_json', '')),
            'model_output':  output,
            'valid_json':    valid_json,
            'n_steps':       n_steps,
            'support_level': support,
        })
        time.sleep(sleep)

    path = os.path.join(RESULTS_DIR, output_filename)
    pd.DataFrame(results).to_csv(path, index=False, encoding='utf-8')

    df_r = pd.DataFrame(results)
    print(f"\n\n── {model_name} Results ──────────────────────")
    print(f"Valid JSON      : {df_r['valid_json'].sum()} ({df_r['valid_json'].mean()*100:.1f}%)")
    print(f"Steps in [1-5]  : {((df_r['n_steps']>=1) & (df_r['n_steps']<=5)).sum()}")
    print(f"Mean steps/card : {df_r[df_r['valid_json']]['n_steps'].mean():.2f}")
    print(f"Saved: {path}")

    return df_r

# Cell 5 — Run all baselines

In [ ]:
# Baseline 1: Qwen3.8 27B (mid-size, non-thinking mode)
df_qwen27b = run_baseline(
    model_id="qwen/qwen3.8-27b",
    model_name="Zero-shot Qwen3.8 27B",
    output_filename="test_outputs_qwen27b_zeroshot.csv",
    extra_body={"reasoning_effort": "none"},
    sleep=3.0
)

# Baseline 2: GPT-OSS 20B
df_gptoss20b = run_baseline(
    model_id="openai/gpt-oss-20b",
    model_name="Zero-shot GPT-OSS 20B",
    output_filename="test_outputs_gptoss20b_zeroshot.csv"
)

# Baseline 3: GPT-OSS 120B (large upper bound)
df_gptoss120b = run_baseline(
    model_id="openai/gpt-oss-120b",
    model_name="Zero-shot GPT-OSS 120B",
    output_filename="test_outputs_gptoss120b_zeroshot.csv",
    sleep=0.5
)

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/VisualStep/results/test_outputs_gptoss120b_zeroshot.csv")
print(f"Total rows: {len(df)}")
print(f"Valid JSON: {df['valid_json'].sum()}")
print(f"NaN outputs: {df['model_output'].isna().sum()}")
print(f"\nLast 3 outputs:")
print(df['model_output'].tail(3).values)

# Cell 6 — BLEU/ROUGE for Qwen3.8 27B

In [ ]:
# Cell 6 — BLEU/ROUGE for Qwen3.8 27B
!pip install rouge-score sacrebleu -q

import json
import pandas as pd
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU

def compute_metrics(df_results, model_name):
    def extract_actions(json_str):
        try:
            data = json.loads(json_str)
            return " ".join(
                f"{s.get('action','')} {s.get('detail','')}".strip()
                for s in data.get("steps", [])
            )
        except:
            return ""

    hyps, refs = [], []
    for _, row in df_results.iterrows():
        h = extract_actions(str(row["model_output"]))
        r = extract_actions(str(row["gold_output"]))
        if h and r:
            hyps.append(h)
            refs.append(r)

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )
    r1 = r2 = rL = 0
    for h, r in zip(hyps, refs):
        s = scorer.score(r, h)
        r1 += s["rouge1"].fmeasure
        r2 += s["rouge2"].fmeasure
        rL += s["rougeL"].fmeasure
    n = len(hyps)

    bleu = BLEU(effective_order=True)
    bleu_score = bleu.corpus_score(hyps, [refs]).score

    print(f"\n── {model_name} ──────────────────────────")
    print(f"Valid pairs : {n}")
    print(f"ROUGE-1     : {r1/n:.4f}")
    print(f"ROUGE-2     : {r2/n:.4f}")
    print(f"ROUGE-L     : {rL/n:.4f}")
    print(f"BLEU-4      : {bleu_score:.4f}")
    return {"model": model_name, "rouge1": r1/n, "rouge2": r2/n,
            "rougeL": rL/n, "bleu4": bleu_score}

# Load Qwen results
df_qwen = pd.read_csv(
    "/content/drive/MyDrive/VisualStep/results/test_outputs_qwen27b_zeroshot.csv"
)

# Compute metrics
result = compute_metrics(df_qwen, "Zero-shot Qwen3.8 27B")
print(f"\nDone.")